# P&ID Stage 2 — encoder one-shot toàn cục trên Colab (GPU)

Train **một** encoder ResNet-34 + ArcFace trên **cả 32 class** DigitizePID (`fgc.global_encoder=true`),
rồi dựng prototype từ template ISO 10628-2 (GĐ2). Thứ tự (docs/PLAN_STAGE2_ONESHOT.md, Phase B):

| Bước | Cell | Ra file trên Drive (`runs/encoders/`) |
|---|---|---|
| R1a — encoder **leave-out** (bằng chứng chính, protocol P1) | 5 | `__global_lo__.pt` |
| R1b — encoder đủ 32 class | 6 | `__global__.pt` |
| R2 — prototype ISO + separation | 7 | `iso_prototypes_lo.pt`, `iso_prototypes.pt` |

R1a chạy TRƯỚC: nếu hết quota Colab thì vẫn có bằng chứng P1.

**Trước khi chạy**: B5 xong — `oneshot.holdout.h_in/h_out` trong `configs/config.yaml` đã điền
và đã push. Cell 4 dừng lại nếu danh sách trống. **Runtime**: GPU (T4 đủ).

## 1. GPU + mount Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
# Thư mục dự án trên Drive phải chứa data/canonical/. Notebook Stage 1 dùng MyDrive/pid, bản
# Stage 2 cũ dùng MyDrive/PID_Recognition -> thử cả hai; sửa DRIVE_ROOT nếu ở chỗ khác.
DRIVE_ROOT = next((d for d in ('/content/drive/MyDrive/pid', '/content/drive/MyDrive/PID_Recognition')
                   if os.path.isdir(f'{d}/data/canonical')), None)
assert DRIVE_ROOT, 'Không thấy <DRIVE_ROOT>/data/canonical — sửa DRIVE_ROOT bằng tay'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_DIR  = f'{DRIVE_ROOT}/runs'
ENC_DIR   = f'{RUNS_DIR}/encoders'      # encoder + prototype đã đặt tên rõ ràng
os.makedirs(ENC_DIR, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('canonical images:', len(os.listdir(f'{DATA_ROOT}/canonical/images')))

## 2. Code + deps

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'   # <-- sửa
BRANCH   = 'feat/prepare-for-colab-stage-2'
%cd /content
![ -d pid ] || git clone -b $BRANCH $REPO_URL pid
%cd /content/pid
!git fetch -q && git checkout -q $BRANCH && git pull -q && git log --oneline -1
!pip install -q uv
# Colab đã có torch CUDA -> không cài torch. run_pipeline.py import mọi stage (cả detector),
# nên cần đủ ultralytics/sahi/ensemble-boxes dù Stage 2 không dùng tới.
!uv pip install --system -q ultralytics sahi ensemble-boxes albumentations timm hydra-core omegaconf \
    pycocotools opencv-python-headless pillow pandas scikit-learn matplotlib tqdm pyyaml
import sys
os.environ['PYTHONPATH'] = '/content/pid/src'
sys.path.insert(0, '/content/pid/src')
REPO_DATA = '/content/pid/data/raw'     # template ISO + vocab.csv nằm trong git, không ở Drive

## 3. Crop toàn cục — đủ 32 class

**Bẫy im lặng**: `derived/crops_fgc/` và `crops_fgc.zip` cũ trên Drive chỉ có **28 class** (cắt theo
`fgc_group`). Notebook này KHÔNG BAO GIỜ đọc chúng: crop toàn cục được cắt ở local rồi lưu thành
`crops_fgc_global.zip` (tên khác). Cắt vào thư mục cũ cũng không được — tên file đánh số khác
nhau giữa hai chế độ nên sẽ ra crop trùng.

Phiên đầu: copy ảnh canonical (~1.2 GB) về local, cắt, zip lên Drive. Phiên sau: chỉ giải nén.

In [ ]:
import glob, shutil, time
from utils.classmap import load_classmap
LOCAL_DATA_ROOT = '/content/data'
LOCAL_CROPS = f'{LOCAL_DATA_ROOT}/derived/crops_fgc'
GLOBAL_ZIP  = f'{DATA_ROOT}/derived/crops_fgc_global.zip'
t0 = time.time()
os.makedirs(f'{LOCAL_DATA_ROOT}/canonical', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_ROOT}/derived', exist_ok=True)
for f in ('classes.yaml', 'manifest.csv', 'annotations.coco.json', 'digitizepid_splits.json'):
    if os.path.exists(f'{DATA_ROOT}/canonical/{f}'):
        shutil.copy2(f'{DATA_ROOT}/canonical/{f}', f'{LOCAL_DATA_ROOT}/canonical/{f}')

if os.path.exists(GLOBAL_ZIP):
    !unzip -q -o "$GLOBAL_ZIP" -d "$LOCAL_DATA_ROOT/derived/"
else:
    !rsync -a "$DATA_ROOT/canonical/images/" "$LOCAL_DATA_ROOT/canonical/images/"
    for step, f in (('classes', 'classes.yaml'), ('manifest', 'manifest.csv')):
        if not os.path.exists(f'{LOCAL_DATA_ROOT}/canonical/{f}'):
            !cd /content/pid && python src/run_pipeline.py prep --step $step data_root=$LOCAL_DATA_ROOT
    !cd /content/pid && python src/run_pipeline.py prep --step crops data_root=$LOCAL_DATA_ROOT fgc.global_encoder=true
    !cd "$LOCAL_DATA_ROOT/derived" && zip -q -r crops_fgc_global.zip crops_fgc && cp crops_fgc_global.zip "$GLOBAL_ZIP"
    print('Đã lưu', GLOBAL_ZIP)
print(f'Crops local sau {time.time()-t0:.0f}s')

# Phải đủ ĐÚNG mọi fine class của classes.yaml ở cả hai split (32 với DigitizePID).
CM = load_classmap(f'{LOCAL_DATA_ROOT}/canonical/classes.yaml')
for split in ('train', 'val'):
    present = set(os.listdir(f'{LOCAL_CROPS}/{split}'))
    n = len(glob.glob(f'{LOCAL_CROPS}/{split}/*/*.png'))
    print(f'  {split}: {n} crops, {len(present)} fine classes')
    missing = sorted(set(CM.fine_classes) - present)
    assert not missing, f'{split} thiếu class {missing} — zip cũ 28 class? xoá {GLOBAL_ZIP} rồi chạy lại'
    assert len(present) == len(CM.fine_classes) == 32, sorted(present)

## 4. Danh sách giữ ra H (từ config, KHÔNG gõ tay)

Đọc `oneshot.holdout` trong `configs/config.yaml` (điền ở B5). R1a train với
`fgc.exclude_classes = h_in + h_out`. Tên sai -> lỗi ngay tại đây, trước khi tốn GPU.

In [ ]:
from omegaconf import OmegaConf
from utils.classmap import GLOBAL_GROUP, training_fine_classes
from utils.oneshot import holdout_classes
H_IN, H_OUT = holdout_classes(OmegaConf.load('/content/pid/configs/config.yaml'))
H = H_IN + H_OUT
assert H, 'oneshot.holdout trống: làm B5 (chọn H_in/H_out), sửa configs/config.yaml, push, rồi chạy lại cell 2'
KEPT = training_fine_classes(CM, GLOBAL_GROUP, H)       # tên không có trong classes.yaml -> ValueError
EXCLUDE = '[' + ','.join(H) + ']'
print(f'H_in  ({len(H_IN)}):', H_IN)
print(f'H_out ({len(H_OUT)}):', H_OUT)
print(f'R1a train trên {len(KEPT)}/{len(CM.fine_classes)} class')

## 5. R1a — encoder leave-out → `__global_lo__.pt`

Checkpoint ghi thẳng vào Drive (`runs/r1a/<ts>/fgc_checkpoints/__global__.pt`, lưu mỗi khi val_acc
tăng). Cell 5b kiểm checkpoint rồi copy thành `encoders/__global_lo__.pt`.

In [ ]:
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1a \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=60 fgc.batch=128 \
    "fgc.exclude_classes=$EXCLUDE"

**5a. Resume R1a** (nếu Colab ngắt) — tự lấy run R1a mới nhất; CÙNG danh sách H (code chặn nếu khác).

In [ ]:
R1A_LAST = sorted(glob.glob(f'{RUNS_DIR}/r1a/*/fgc_checkpoints/__global__.pt'), key=os.path.getmtime)[-1]
R1A_RESUME = os.path.dirname(R1A_LAST)
print('resume từ', R1A_RESUME)
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1a \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=60 fgc.batch=128 \
    "fgc.exclude_classes=$EXCLUDE" fgc.resume_from=$R1A_RESUME

**5b. Kiểm + đặt tên** — encoder KHÔNG được có cột nào của H.

In [ ]:
import torch
def finalize(run_glob, out_name, expect_excluded):
    ck = sorted(glob.glob(run_glob), key=os.path.getmtime)[-1]
    st = torch.load(ck, map_location='cpu', weights_only=False)
    fine = list(st['fine_classes'])
    assert not set(expect_excluded) & set(fine), f'RÒ RỈ: {sorted(set(expect_excluded) & set(fine))} có trong encoder'
    assert len(fine) == len(CM.fine_classes) - len(expect_excluded), (len(fine), len(expect_excluded))
    assert sorted(st.get('excluded_classes', [])) == sorted(expect_excluded), st.get('excluded_classes')
    assert st['model']['head.weight'].shape[0] == len(fine)
    shutil.copy2(ck, f'{ENC_DIR}/{out_name}')
    print(f'{ck}\n  -> {ENC_DIR}/{out_name}: {len(fine)} class, epoch {st["epoch"]+1}, val_acc {st["val_acc"]:.4f}')

finalize(f'{RUNS_DIR}/r1a/*/fgc_checkpoints/__global__.pt', '__global_lo__.pt', H)

## 6. R1b — encoder đủ 32 class → `__global__.pt`

Cho pipeline end-to-end, P2 và P3. `fgc.exclude_classes=[]` ghi rõ để không thừa hưởng nhầm.

In [ ]:
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1b \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=60 fgc.batch=128 \
    "fgc.exclude_classes=[]"

**6a. Resume R1b** (nếu Colab ngắt).

In [ ]:
R1B_LAST = sorted(glob.glob(f'{RUNS_DIR}/r1b/*/fgc_checkpoints/__global__.pt'), key=os.path.getmtime)[-1]
R1B_RESUME = os.path.dirname(R1B_LAST)
print('resume từ', R1B_RESUME)
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1b \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=60 fgc.batch=128 \
    "fgc.exclude_classes=[]" fgc.resume_from=$R1B_RESUME

**6b. Kiểm + đặt tên.**

In [ ]:
finalize(f'{RUNS_DIR}/r1b/*/fgc_checkpoints/__global__.pt', '__global__.pt', [])

## 7. R2 — prototype ISO + separation (GĐ2)

Mỗi encoder có sẵn trong `encoders/` -> một prototype directory. Đọc dòng
`prototype separation: mean off-diagonal cos=…, max=… (closest pair: A vs B)` và ghi số vào HANDOFF
(bảng B4 trong plan: max < ~0.8 tốt; mean ≥ ~0.6 = encoder không transfer -> GĐ3).

In [ ]:
for enc, out in (('__global_lo__.pt', 'iso_prototypes_lo.pt'), ('__global__.pt', 'iso_prototypes.pt')):
    if not os.path.exists(f'{ENC_DIR}/{enc}'):
        print('bỏ qua (chưa có)', enc); continue
    print(f'===== {enc} -> {out}')
    !cd /content/pid && python src/run_pipeline.py prototypes run_dir=$RUNS_DIR/r2 \
        oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$out \
        oneshot.symbols_dir=$REPO_DATA/iso_symbols_clean oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv \
        2>&1 | grep -v "^Warning"

## 8. Eval — CHƯA dùng được với encoder toàn cục

`evaluation.py` lặp theo `fgc_groups` và tìm `<group>.pt`, nên với `__global__.pt` nó **bỏ qua im
lặng** (không có `valve.pt`…) và báo như thể không có FGC. Eval one-shot P1/P2/P3 là B6b — cell dưới
để comment cho tới khi B6b xong.

In [ ]:
# !cd /content/pid && python src/run_pipeline.py evaluate \
#     data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR eval.mode=det_plus_fgc eval.split=val \
#     fgc.metric_learning=true fgc.ckpt_dir=<fgc_checkpoints>